# DriveAware: Model A training (driver activity classifier)

**Runs in Google Colab with a GPU** (Runtime → Change runtime type → T4 GPU). Mahitha runs it; Bindu evaluates the results.

**Input:** `DriveAware/Frames/` in Drive, made by `scripts/extract_frames.py` (224×224 crops + `labels.csv`).
**Output:** `DriveAware/Models/model_a_v1.tflite`, plus metrics, curves and confusion matrices in `DriveAware/Results/model_a/`.

What it trains, in the order the guidelines need for the report:
1. **CLIP ViT-B/32 zero-shot**: pretrained, no training (baseline 1)
2. **Frozen backbone**: only the last layer trained on our data (baseline 2)
3. **Fully fine-tuned**: all layers adapted; this is **Model A**

Path A uses **MediaPipe Model Maker, EfficientNet-Lite0** (the proposal). If Model Maker won't install on this Colab
(it needs Python ≤ 3.12), the notebook automatically uses **path B: Keras MobileNetV3-Small** (the plan's fallback, task #17).

**Rules:** pick the best run by **validation** score, never test. Test = held-out members only (Mahitha, Vanessa).
Log every run (the last cell appends to `Results/model_a/model_a_runs.csv`). Change one setting at a time.

In [ ]:
# ===== 1. Settings: change one at a time, re-run from here =====
DRIVE_ROOT  = "/content/drive/MyDrive/DriveAware"   # if it's under "Shared with me", add a shortcut to My Drive first
FRAMES_DIR  = f"{DRIVE_ROOT}/Frames"                # from scripts/extract_frames.py
RESULTS_DIR = f"{DRIVE_ROOT}/Results/model_a"
MODELS_DIR  = f"{DRIVE_ROOT}/Models"
RUN_NAME    = "run01"          # change for every run, e.g. run02_lr5e-4
LABEL_LEVEL = "app_class"      # "app_class" (5 classes the app uses) or "subclass" (10)
EPOCHS      = 15
BATCH_SIZE  = 32
LEARNING_RATE = 1e-3           # head; fine-tuning uses LEARNING_RATE / 10
FROZEN_EPOCHS = 5              # baseline 2 (frozen backbone)
EARLY_STOP_PATIENCE = 3
MIN_TRAIN_PER_CLASS = 20       # classes with fewer training images are dropped (and reported)
FORCE_KERAS = False            # True = skip Model Maker, use MobileNetV3-Small
SEED = 663

In [ ]:
# ===== 2. Mount Drive =====
from google.colab import drive
drive.mount("/content/drive")
import os
assert os.path.exists(f"{FRAMES_DIR}/labels.csv"), f"No labels.csv in {FRAMES_DIR}: run scripts/extract_frames.py first"
os.makedirs(RESULTS_DIR, exist_ok=True); os.makedirs(MODELS_DIR, exist_ok=True)
print("ok:", FRAMES_DIR)

In [ ]:
# ===== 3. Load labels.csv and check it =====
import pandas as pd, numpy as np
df = pd.read_csv(f"{FRAMES_DIR}/labels.csv")
df["path"] = FRAMES_DIR + "/" + df["frame_path"]
missing = ~df["path"].map(os.path.exists)
if missing.any():
    print(f"! {missing.sum()} rows point to missing images; dropped")
df = df[~missing].copy()
df["label"] = df[LABEL_LEVEL]

HELD_OUT = {"mahitha", "vanessa"}
team = df["source"].eq("team")
leak = team & df["subject"].str.lower().isin(HELD_OUT) & df["split"].ne("test")
assert not leak.any(), f"{leak.sum()} held-out frames are not in test: fix labels.csv split column"

counts = df.pivot_table(index="label", columns="split", values="path", aggfunc="count", fill_value=0)
print(counts)
train_counts = df[df.split == "train"]["label"].value_counts()
CLASSES = sorted(train_counts[train_counts >= MIN_TRAIN_PER_CLASS].index)
dropped = sorted(set(df["label"]) - set(CLASSES))
print("\nclasses trained:", CLASSES)
if dropped:
    print("not enough training images (reported as 'unseen' in test):", dropped)

In [ ]:
# ===== 4. Folder layout for training (copied to the Colab disk: reading Drive through symlinks is very slow) =====
import shutil
DATA = "/content/data"
shutil.rmtree(DATA, ignore_errors=True)
for split in ("train", "val", "test"):
    for c in CLASSES:
        os.makedirs(f"{DATA}/{split}/{c}", exist_ok=True)
n = 0
for r in df[df["label"].isin(CLASSES)].itertuples():
    dst = f"{DATA}/{r.split}/{r.label}/{n:06d}_{os.path.basename(r.path)}"
    shutil.copyfile(r.path, dst); n += 1
print(n, "images copied into", DATA)

In [ ]:
# ===== 5. Install Model Maker (path A) or fall back to Keras (path B) =====
import sys, subprocess
USE_MM = False
if not FORCE_KERAS and sys.version_info < (3, 13):
    r = subprocess.run([sys.executable, "-m", "pip", "install", "-q", "mediapipe-model-maker"], capture_output=True, text=True)
    try:
        from mediapipe_model_maker import image_classifier  # noqa: F401
        USE_MM = True
    except Exception as e:
        print("Model Maker unavailable:", str(e)[:200])
print("Python", sys.version.split()[0], "→ path", "A: Model Maker EfficientNet-Lite0" if USE_MM else "B: Keras MobileNetV3-Small")
ARCH = "EfficientNet-Lite0 (Model Maker)" if USE_MM else "MobileNetV3-Small (Keras)"

In [ ]:
# ===== 6. Train: frozen backbone (baseline 2) and fully fine-tuned (Model A) =====
import json, time
import tensorflow as tf
tf.keras.utils.set_random_seed(SEED)
HIST = {}

if USE_MM:
    from mediapipe_model_maker import image_classifier
    train_data = image_classifier.Dataset.from_folder(f"{DATA}/train")
    val_data = image_classifier.Dataset.from_folder(f"{DATA}/val")
    def mm_train(fine_tune, epochs, export_dir):
        hp = image_classifier.HParams(export_dir=export_dir, epochs=epochs, batch_size=BATCH_SIZE,
                                      learning_rate=LEARNING_RATE if not fine_tune else LEARNING_RATE / 10,
                                      do_fine_tuning=fine_tune, shuffle=True)
        opts = image_classifier.ImageClassifierOptions(
            supported_model=image_classifier.SupportedModels.EFFICIENTNET_LITE0, hparams=hp,
            model_options=image_classifier.ModelOptions(dropout_rate=0.2))
        m = image_classifier.ImageClassifier.create(train_data=train_data, validation_data=val_data, options=opts)
        m.export_model()   # export_dir/model.tflite (with metadata, ready for the MediaPipe Image Classifier)
        return m
    t0 = time.time(); frozen = mm_train(False, FROZEN_EPOCHS, "/content/out_frozen")
    t1 = time.time(); finetuned = mm_train(True, EPOCHS, "/content/out_finetuned")
    TFLITE = {"frozen": "/content/out_frozen/model.tflite", "finetuned": "/content/out_finetuned/model.tflite"}
    print(f"frozen {t1-t0:.0f}s, fine-tuned {time.time()-t1:.0f}s")
else:
    IMG = (224, 224)
    ds = {s: tf.keras.utils.image_dataset_from_directory(f"{DATA}/{s}", image_size=IMG, batch_size=BATCH_SIZE,
                                                         class_names=CLASSES, shuffle=(s == "train"), seed=SEED)
          for s in ("train", "val")}
    aug = tf.keras.Sequential([tf.keras.layers.RandomFlip("horizontal"), tf.keras.layers.RandomRotation(0.05),
                               tf.keras.layers.RandomZoom(0.1), tf.keras.layers.RandomContrast(0.2),
                               tf.keras.layers.RandomBrightness(0.2)])
    base = tf.keras.applications.MobileNetV3Small(input_shape=IMG + (3,), include_top=False, pooling="avg",
                                                  weights="imagenet", include_preprocessing=True)
    inp = tf.keras.Input(IMG + (3,)); x = aug(inp); x = base(x, training=False)
    x = tf.keras.layers.Dropout(0.2)(x); out = tf.keras.layers.Dense(len(CLASSES), activation="softmax")(x)
    model = tf.keras.Model(inp, out)
    stop = lambda: [tf.keras.callbacks.EarlyStopping(monitor="val_accuracy", patience=EARLY_STOP_PATIENCE,
                                                     restore_best_weights=True)]
    base.trainable = False
    model.compile(optimizer=tf.keras.optimizers.Adam(LEARNING_RATE), loss="sparse_categorical_crossentropy", metrics=["accuracy"])
    HIST["frozen"] = model.fit(ds["train"], validation_data=ds["val"], epochs=FROZEN_EPOCHS, callbacks=stop()).history
    frozen = tf.keras.models.clone_model(model); frozen.set_weights(model.get_weights())
    base.trainable = True
    for layer in base.layers:                      # keep BatchNorm frozen while fine-tuning small data
        if isinstance(layer, tf.keras.layers.BatchNormalization): layer.trainable = False
    model.compile(optimizer=tf.keras.optimizers.Adam(LEARNING_RATE / 10), loss="sparse_categorical_crossentropy", metrics=["accuracy"])
    HIST["finetuned"] = model.fit(ds["train"], validation_data=ds["val"], epochs=EPOCHS, callbacks=stop()).history
    finetuned = model
    def to_tflite(m, path):
        conv = tf.lite.TFLiteConverter.from_keras_model(m)
        conv.optimizations = [tf.lite.Optimize.DEFAULT]; conv.target_spec.supported_types = [tf.float16]
        buf = conv.convert()
        # Attach labels + normalization so the app's MediaPipe ImageClassifier can load it
        # (Model Maker exports already have this). MobileNetV3 here takes raw 0-255 pixels.
        try:
            subprocess.run([sys.executable, "-m", "pip", "install", "-q", "mediapipe"], check=False)
            from mediapipe.tasks.python.metadata.metadata_writers import image_classifier as ic_writer, metadata_writer
            writer = ic_writer.MetadataWriter.create(buf, input_norm_mean=[0.0], input_norm_std=[1.0],
                                                     labels=metadata_writer.Labels().add(CLASSES))
            buf, _ = writer.populate()
        except Exception as e:   # not available on Colab's Python 3.13: the app reads model_a_labels.txt instead
            print("TFLite metadata not attached (", str(e)[:80], "); save labels.txt next to the model")
        open(path, "wb").write(buf); return path
    TFLITE = {"frozen": to_tflite(frozen, "/content/frozen.tflite"),
              "finetuned": to_tflite(finetuned, "/content/finetuned.tflite")}
print(TFLITE)

In [ ]:
# ===== 7. Learning curves (train vs validation loss and accuracy) =====
import matplotlib.pyplot as plt
if HIST:
    fig, ax = plt.subplots(1, 2, figsize=(11, 4))
    for name, h in HIST.items():
        ax[0].plot(h["loss"], label=f"{name} train"); ax[0].plot(h["val_loss"], "--", label=f"{name} val")
        ax[1].plot(h["accuracy"], label=f"{name} train"); ax[1].plot(h["val_accuracy"], "--", label=f"{name} val")
    ax[0].set_title("loss"); ax[1].set_title("accuracy"); for a in ax: a.set_xlabel("epoch"); a.legend()
    plt.tight_layout(); plt.savefig(f"{RESULTS_DIR}/{RUN_NAME}_curves.png", dpi=120); plt.show()
else:
    print("Model Maker prints per-epoch loss/accuracy above; screenshot that output for the report.")

In [ ]:
# ===== 8. Predict on val + test with every model (same code for all) =====
from PIL import Image
def load_rgb(p): return np.asarray(Image.open(p).convert("RGB").resize((224, 224)))

if USE_MM:
    import mediapipe as mp
    from mediapipe.tasks.python import vision, BaseOptions
    def predictor(tflite_path):
        clf = vision.ImageClassifier.create_from_options(vision.ImageClassifierOptions(
            base_options=BaseOptions(model_asset_path=tflite_path), max_results=len(CLASSES)))
        def predict(paths):
            out = []
            for p in paths:
                cats = clf.classify(mp.Image(image_format=mp.ImageFormat.SRGB, data=load_rgb(p))).classifications[0].categories
                scores = {c.category_name: c.score for c in cats}
                out.append([scores.get(c, 0.0) for c in CLASSES])
            return np.array(out)
        return predict
else:
    def predictor(tflite_path):
        it = tf.lite.Interpreter(model_path=tflite_path); it.allocate_tensors()
        i, o = it.get_input_details()[0], it.get_output_details()[0]
        def predict(paths):
            out = []
            for p in paths:
                it.set_tensor(i["index"], load_rgb(p)[None].astype(np.float32)); it.invoke()
                out.append(it.get_tensor(o["index"])[0])
            return np.array(out)
        return predict

evaluate_on = df[df.split.isin(["val", "test"])].copy()
PRED = {}
for name, path in TFLITE.items():
    probs = predictor(path)(evaluate_on["path"].tolist())
    PRED[name] = probs
    print(name, "done")

In [ ]:
# ===== 9. Baseline 1: CLIP ViT-B/32 zero-shot (no training) =====
import torch
from transformers import CLIPModel, CLIPProcessor
dev = "cuda" if torch.cuda.is_available() else "cpu"
clip = CLIPModel.from_pretrained("openai/clip-vit-base-patch32").to(dev).eval()
proc = CLIPProcessor.from_pretrained("openai/clip-vit-base-patch32")
PROMPTS = {
    "safe_driving": "a photo of a driver with both hands on the steering wheel looking at the road",
    "phone_use": "a photo of a driver using a mobile phone",
    "drinking": "a photo of a driver drinking from a cup or bottle",
    "smoking": "a photo of a driver smoking a cigarette",
    "face_touch": "a photo of a driver touching or rubbing their face",
    "safe": "a photo of a driver with both hands on the steering wheel looking at the road",
    "texting": "a photo of a driver texting on a phone", "phone_ear": "a photo of a driver talking on a phone held to the ear",
    "phone_in_hand": "a photo of a driver holding a phone in their hand", "drinking_cup": "a photo of a driver drinking from a cup",
    "drinking_bottle": "a photo of a driver drinking from a bottle", "eye_rub": "a photo of a driver rubbing their eyes",
    "face_scratch": "a photo of a driver scratching their face", "hand_forehead": "a photo of a driver with a hand on their forehead",
}
with torch.no_grad():
    texts = [PROMPTS[c] for c in CLASSES]
    probs = []
    paths = evaluate_on["path"].tolist()
    for k in range(0, len(paths), 64):
        ims = [Image.open(p).convert("RGB") for p in paths[k:k + 64]]
        out = clip(**proc(text=texts, images=ims, return_tensors="pt", padding=True).to(dev))
        probs.append(out.logits_per_image.softmax(dim=-1).cpu().numpy())   # works on old and new transformers
PRED["clip_zeroshot"] = np.concatenate(probs)
print("CLIP done")

In [ ]:
# ===== 10. Metrics: accuracy, per-class P/R/F1, macro-F1, confusion matrices, breakdowns =====
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix, ConfusionMatrixDisplay
SUB2APP = {"safe": "safe_driving", "phone_ear": "phone_use", "texting": "phone_use", "phone_in_hand": "phone_use",
           "drinking_cup": "drinking", "drinking_bottle": "drinking", "smoking": "smoking",
           "eye_rub": "face_touch", "face_scratch": "face_touch", "hand_forehead": "face_touch"}

def scores(split, name):
    m = (evaluate_on["split"] == split).to_numpy()
    sub = evaluate_on[m].copy()
    sub["pred"] = [CLASSES[i] for i in PRED[name][m].argmax(1)]
    sub["conf"] = PRED[name][m].max(1)
    seen = sub["label"].isin(CLASSES)
    return sub, seen

rows, reports = [], {}
for name in ["clip_zeroshot", "frozen", "finetuned"]:
    for split in ["val", "test"]:
        sub, seen = scores(split, name)
        s = sub[seen]
        if len(s) == 0: continue
        rows.append({"model": name, "split": split, "n": len(s), "unseen_class_n": int((~seen).sum()),
                     "accuracy": accuracy_score(s.label, s.pred),
                     "macro_f1": f1_score(s.label, s.pred, labels=CLASSES, average="macro", zero_division=0)})
        if split == "test":
            reports[name] = classification_report(s.label, s.pred, labels=CLASSES, zero_division=0, output_dict=True)
summary = pd.DataFrame(rows).round(3)
print(summary.to_string(index=False))
summary.to_csv(f"{RESULTS_DIR}/{RUN_NAME}_summary.csv", index=False)
if "finetuned" not in reports:
    raise SystemExit("No test frames yet: add the held-out clips (Team_test_only) to extract_frames.py --team and rerun it.")
pd.DataFrame(reports["finetuned"]).T.round(3).to_csv(f"{RESULTS_DIR}/{RUN_NAME}_finetuned_per_class.csv")

# confusion matrices on test (Model A), at the label level and mapped to the 5 app classes
sub, seen = scores("test", "finetuned"); s = sub[seen]
fig, ax = plt.subplots(1, 2, figsize=(13, 5))
ConfusionMatrixDisplay(confusion_matrix(s.label, s.pred, labels=CLASSES), display_labels=CLASSES).plot(ax=ax[0], xticks_rotation=45, colorbar=False)
ax[0].set_title(f"Model A test ({LABEL_LEVEL})")
app = sorted(set(SUB2APP.values()))
to_app = (lambda v: SUB2APP.get(v, v))
ConfusionMatrixDisplay(confusion_matrix(s.label.map(to_app), s.pred.map(to_app), labels=app), display_labels=app).plot(ax=ax[1], xticks_rotation=45, colorbar=False)
ax[1].set_title("Model A test (5 app classes)")
plt.tight_layout(); plt.savefig(f"{RESULTS_DIR}/{RUN_NAME}_confusion.png", dpi=120); plt.show()

# 10 recorded subclasses (rows) x predicted app class (columns), on ALL test frames, including
# classes the model never saw in training (those rows show what it predicts instead).
allt, _ = scores("test", "finetuned")
SUBS = [c for c in ["safe", "phone_ear", "texting", "phone_in_hand", "drinking_cup", "drinking_bottle",
                    "smoking", "eye_rub", "face_scratch", "hand_forehead"] if c in set(allt["subclass"])]
sub_cm = pd.crosstab(pd.Categorical(allt["subclass"], categories=SUBS),
                     pd.Categorical(allt["pred"].map(to_app), categories=app), dropna=False)
sub_cm.index.name, sub_cm.columns.name = "recorded subclass", "predicted app class"
sub_cm.to_csv(f"{RESULTS_DIR}/{RUN_NAME}_confusion_subclass.csv")
fig, ax = plt.subplots(figsize=(8, 6))
ax.imshow(sub_cm.values, cmap="Blues")
ax.set_xticks(range(len(app)), app, rotation=45, ha="right"); ax.set_yticks(range(len(SUBS)), SUBS)
for i in range(len(SUBS)):
    for j in range(len(app)):
        ax.text(j, i, int(sub_cm.values[i, j]), ha="center", va="center", fontsize=8)
ax.set_xlabel("predicted app class"); ax.set_ylabel("recorded subclass")
ax.set_title(f"Model A test: 10 subclasses x app class ({RUN_NAME})")
plt.tight_layout(); plt.savefig(f"{RESULTS_DIR}/{RUN_NAME}_confusion_subclass.png", dpi=120); plt.show()

# breakdown by lighting and eyewear (team frames: condition like day-sun)
t = s[s.condition.fillna("").str.contains("-")].copy()
if len(t):
    t["lighting"] = t.condition.str.split("-").str[0]; t["eyewear"] = t.condition.str.split("-").str[1]
    t["correct"] = t.label == t.pred
    br = pd.concat([t.groupby("lighting").correct.agg(["count", "mean"]).assign(group="lighting"),
                    t.groupby("eyewear").correct.agg(["count", "mean"]).assign(group="eyewear")])
    br = br.rename(columns={"mean": "accuracy"}).round(3); print(br)
    br.to_csv(f"{RESULTS_DIR}/{RUN_NAME}_by_condition.csv")
if (~seen).any():
    print(f"\n{(~seen).sum()} test frames belong to classes with no training data (unseen_class_n):")
    print(sub[~seen].groupby(["label", "subclass"]).size().to_string(), "\n→ reported, not scored")

In [ ]:
# ===== 11. Success and failure examples (for the report) =====
sub, seen = scores("test", "finetuned"); s = sub[seen]
ok, bad = s[s.label == s.pred].sample(min(8, (s.label == s.pred).sum()), random_state=SEED), \
          s[s.label != s.pred].sample(min(8, (s.label != s.pred).sum()), random_state=SEED)
for title, part in [("success", ok), ("failure", bad)]:
    if not len(part): continue
    fig, axs = plt.subplots(1, len(part), figsize=(2.6 * len(part), 3))
    for a, r in zip(np.atleast_1d(axs), part.itertuples()):
        a.imshow(Image.open(r.path)); a.axis("off")
        a.set_title(f"true {r.label}\npred {r.pred} {r.conf:.2f}", fontsize=8, color="green" if title == "success" else "red")
    plt.tight_layout(); plt.savefig(f"{RESULTS_DIR}/{RUN_NAME}_{title}_examples.png", dpi=120); plt.show()
bad.drop(columns=["path"]).to_csv(f"{RESULTS_DIR}/{RUN_NAME}_failures.csv", index=False)
print("Add a one-line reason for each failure in the report (e.g. phone hidden by hand, night glare).")

In [ ]:
# ===== 12. Log the run and save the model =====
best = summary[(summary.model == "finetuned") & (summary.split == "val")].iloc[0]
log = {"run": RUN_NAME, "arch": ARCH, "label_level": LABEL_LEVEL, "classes": "|".join(CLASSES),
       "epochs": EPOCHS, "frozen_epochs": FROZEN_EPOCHS, "batch": BATCH_SIZE, "lr": LEARNING_RATE,
       "train_n": int((df.split == "train").sum()), "val_acc": best.accuracy, "val_macro_f1": best.macro_f1,
       "test_acc": float(summary[(summary.model == "finetuned") & (summary.split == "test")].accuracy.iloc[0])
                   if ((summary.model == "finetuned") & (summary.split == "test")).any() else None,
       "time": time.strftime("%Y-%m-%d %H:%M")}
log_path = f"{RESULTS_DIR}/model_a_runs.csv"
pd.DataFrame([log]).to_csv(log_path, mode="a", header=not os.path.exists(log_path), index=False)
print(pd.read_csv(log_path).to_string(index=False))

shutil.copy(TFLITE["finetuned"], f"{RESULTS_DIR}/{RUN_NAME}_model_a.tflite")
open(f"{RESULTS_DIR}/{RUN_NAME}_labels.txt", "w").write("\n".join(CLASSES))
print(f"\nSaved {RUN_NAME}_model_a.tflite + labels.txt in Results/model_a/.")
print("When this is the BEST run by validation score, copy it to Models/ as model_a_v1.tflite and "
      "its labels file as model_a_labels.txt (the app reads the class names from it), then tell Swathi.")